In [ ]:
# This line installs the latest version of cvxpy, which is needed to use the
# IPOPT solver below. After the next cvxpy release, just importing cvxpy will be
# sufficient.
!pip install git+https://github.com/cvxpy/cvxpy.git

In [ ]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [ ]:
# Problem data.
m = 1 # mass
I = 1 # moment of inertia
g = 10 # gravity acceleration
r = 1 # thrust distance from COM
h = .05 # discretization time step
K = 50 # number of time steps

# Initial state.
x0 = np.array([1, 3, np.pi/8, 0, 0, 0])

In [ ]:
# Right-hand side of rocket's continuous-time dynamics in state-space form.
def f(x, u):
    # ### YOUR CODE HERE
    # ### YOUR CODE HERE
    # ### YOUR CODE HERE
    return cp.hstack([0, 0, 0, 0, 0, 0]) ### MODIFY HERE

In [ ]:
# Variables.
X = cp.Variable((K + 1, 6)) # state at each time step X[0], ..., X[K]
U = cp.Variable((K, 2)) # input at each time step U[0], ..., U[K-1]

# Initialize empty list of constraints.
constraints = []

# Explicit-Euler approximation of rocket dynamics.
for k in range(K):
    constraints.append(X[k + 1] == X[k] + h * f(X[k], U[k]))

# Initial state constraint.
constraints.append(X[0] == x0)

# Final state constraint.
### YOUR CODE HERE
### YOUR CODE HERE

# Limit on thrust angle.
### YOUR CODE HERE
### YOUR CODE HERE

# Constraint preventing the rocket from colliding with the ground before landing.
### YOUR CODE HERE
### YOUR CODE HERE

# Objective function.
obj = h * cp.sum_squares(U)

# Solution initial guess (initial point for the solver).
X.value = np.linspace(x0, 0, K + 1)
U.value = np.zeros((K, 2))

# Solve problem.
prob = cp.Problem(cp.Minimize(obj), constraints)
prob.solve(nlp=True, solver=cp.IPOPT)

In [ ]:
# Pot optimal control inputs.
time_steps = np.arange(K)
plt.plot(time_steps, U[:, 0].value, 'r', label=r'$u_1$')
plt.plot(time_steps, np.abs(U[:, 0].value), 'r--', label=r'$|u_1|$')
plt.plot(time_steps, U[:, 1].value, label=r'$u_2$')
plt.xlabel('Time step')
plt.ylabel(r'Control inputs')
plt.grid()
plt.legend()

In [ ]:
# Animate rocket motion.
fig, ax = plt.subplots()
limits = np.array([
    np.minimum(x0[:2], 0) - 2 * r,
    np.maximum(x0[:2], 0) + 2 * r])
ax.set_xlim(*limits[:, 0])  # adjust animation width to your trajectory
ax.set_ylim(*limits[:, 1])
ax.set_aspect('equal')
ax.grid()

# Rocket and thrust represented as lines.
rocket, = ax.plot([], [], lw=10)
thrust, = ax.plot([], [], color='red', lw=2)
def init():
    rocket.set_data([], [])
    thrust.set_data([], [])
    return rocket, thrust

# Animation update function.
def update(frame):
    q1, q2, q3 = X.value[frame, :3]
    u1, u2 = U.value[frame] / 10
    rocket_x = [q1 + r * np.sin(q3), q1 - r * np.sin(q3)]
    rocket_y = [q2 - r * np.cos(q3), q2 + r * np.cos(q3)]
    thrust_x = [q1 + r * np.sin(q3), q1 + (r + u2) * np.sin(q3) - u1 * np.cos(q3)]
    thrust_y = [q2 - r * np.cos(q3), q2 - (r + u2) * np.cos(q3) - u1 * np.sin(q3)]
    rocket.set_data(rocket_x, rocket_y)
    thrust.set_data(thrust_x, thrust_y)
    return rocket, thrust

# Create animation.
ani = FuncAnimation(fig, update, frames=K, init_func=init, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())